# Session 3: Feature Engineering & Data Scaling

This notebook walks through the full session **cell by cell**, using a small
synthetic "sensor log" database (timestamped IoT-style readings) to make every
technique concrete. Each code cell is preceded by a short explanation of the
method(s) it introduces.

**Roadmap**
1. Advanced Feature Engineering (Time-Series Data)
2. Feature Scaling (Normalization & Standardization)
3. Integration into ML Pipelines


## 0. Setup & Sample Database

We first import the libraries we'll need:

- **`pandas`** — builds and manipulates our tabular "database" (a `DataFrame`).
- **`numpy`** — numerical operations, used for the sin/cos cyclical transforms.
- **`scikit-learn`** (`sklearn`) — provides `StandardScaler`, `MinMaxScaler`,
  `RobustScaler`, `OneHotEncoder`, `OrdinalEncoder`, `ColumnTransformer`, and
  `Pipeline` for the modeling steps later.

We then generate a synthetic **sensor log table** that mimics a real
database export: a raw `timestamp` string column, a numeric `reading`
column (with a couple of outliers on purpose), and a categorical
`device_type` column.


In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, RobustScaler,
    OneHotEncoder, OrdinalEncoder
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

np.random.seed(42)

# Simulate a raw database export: timestamp stored as string, plus a
# numeric reading and a categorical device type. A few outliers are
# injected into 'reading' on purpose so RobustScaler has something to prove.
n = 200
timestamps = pd.date_range("2024-01-01", periods=n, freq="37min")
timestamp_strings = timestamps.strftime("%Y-%m-%d %H:%M:%S")

readings = np.random.normal(loc=50, scale=5, size=n)
outlier_idx = np.random.choice(n, size=5, replace=False)
readings[outlier_idx] += np.random.choice([150, -120], size=5)  # inject outliers

device_types = np.random.choice(["TempSensor", "HumiditySensor", "PressureSensor"], size=n)

df = pd.DataFrame({
    "timestamp_str": timestamp_strings,   # raw datetime STRING, as it would come from a DB/log file
    "reading": readings,
    "device_type": device_types
})

df.head()


,timestamp_str,reading,device_type
0,2024-01-01 00:00:00,52.483571,PressureSensor
1,2024-01-01 00:37:00,49.308678,TempSensor
2,2024-01-01 01:14:00,53.238443,TempSensor
3,2024-01-01 01:51:00,57.615149,TempSensor
4,2024-01-01 02:28:00,48.829233,PressureSensor


## 1. Advanced Feature Engineering (Time-Series Data)

### 1-1. Parsing Temporal Strings

Raw databases and log files almost always store datetimes as **strings**
(e.g. `"2024-01-01 00:37:00"`). Models can't use a string directly — we must
deconstruct it into granular numeric features: `year`, `month`, `day`,
`hour`, `minute`, `weekday`, etc.

Methods used:

- **`pd.to_datetime()`** — parses a string column into pandas' native
  `datetime64` dtype, which unlocks the `.dt` accessor.
- **`Series.dt.year / .dt.month / .dt.day / .dt.hour`** — vectorized
  accessors that pull each temporal component out of a `datetime64` column
  in one pass (much faster than manual string splitting on large tables).


In [2]:
# Convert the raw string column into a real datetime dtype
df["timestamp"] = pd.to_datetime(df["timestamp_str"])

# Deconstruct into granular components using the .dt accessor
df["year"]    = df["timestamp"].dt.year
df["month"]   = df["timestamp"].dt.month
df["day"]     = df["timestamp"].dt.day
df["hour"]    = df["timestamp"].dt.hour
df["minute"]  = df["timestamp"].dt.minute
df["weekday"] = df["timestamp"].dt.weekday  # 0 = Monday ... 6 = Sunday

df[["timestamp_str", "year", "month", "day", "hour", "minute", "weekday"]].head()


,timestamp_str,year,month,day,hour,minute,weekday
0,2024-01-01 00:00:00,2024,1,1,0,0,0
1,2024-01-01 00:37:00,2024,1,1,0,37,0
2,2024-01-01 01:14:00,2024,1,1,1,14,0
3,2024-01-01 01:51:00,2024,1,1,1,51,0
4,2024-01-01 02:28:00,2024,1,1,2,28,0


### 1-2. Feature Extraction Logic (Manual Loop / String-Split Version)

`pd.to_datetime` + `.dt` is the efficient, production way to do this.
But it's worth seeing the **manual loop and string-splitting** approach too,
since that's the technique referenced in the session and the one you'd fall
back on for oddly-formatted logs that pandas can't parse automatically.

Method used:

- **`str.split()`** — splits a string into a list on a delimiter (here, a
  space to separate date from time, then `-` and `:` to split further).
  Looping row-by-row and calling `.split()` is O(n) but far slower than the
  vectorized `.dt` accessor above — shown here for teaching purposes only.


In [3]:
def extract_datetime_parts(ts_string):
    """Manually split a 'YYYY-MM-DD HH:MM:SS' string into its parts.
    Mirrors what pd.to_datetime + .dt does internally, one row at a time."""
    date_part, time_part = ts_string.split(" ")
    y, mo, d = date_part.split("-")
    h, mi, s = time_part.split(":")
    return int(y), int(mo), int(d), int(h), int(mi), int(s)

# Loop over every row and build the feature lists manually
years, months, days, hours, minutes, seconds = [], [], [], [], [], []
for ts in df["timestamp_str"]:
    y, mo, d, h, mi, s = extract_datetime_parts(ts)
    years.append(y); months.append(mo); days.append(d)
    hours.append(h); minutes.append(mi); seconds.append(s)

# Sanity check: manual extraction should match the vectorized .dt columns
manual_check = pd.DataFrame({"manual_hour": hours, "dt_hour": df["hour"]})
assert (manual_check["manual_hour"] == manual_check["dt_hour"]).all()
print("Manual string-split extraction matches pandas .dt extraction ✔")


Manual string-split extraction matches pandas .dt extraction ✔


### 1-3. Handling Cyclical Features (sin/cos Encoding)

A raw `hour` value of `23` and `0` look far apart numerically, even though
they are only 1 hour apart on the clock. The same problem applies to
`month` (December `12` vs January `1`). If we feed these integers straight
into a model, it wrongly assumes 23 and 0 are "distant".

**Solution:** map the cyclical value onto a circle using sine and cosine.
For a feature with period `P` (24 for hours, 12 for months):

```
sin_feature = sin(2 * pi * value / P)
cos_feature = cos(2 * pi * value / P)
```

Using both sin AND cos (not just one) is essential — it guarantees every
point on the cycle maps to a unique `(sin, cos)` pair.

Methods used:

- **`np.sin()` / `np.cos()`** — vectorized trigonometric transforms applied
  to an entire pandas column at once.
- **`np.pi`** — constant used inside the periodic transform.


In [4]:
# Cyclical encoding for HOUR (period = 24)
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

# Cyclical encoding for MONTH (period = 12; months are 1-12, so shift by -1)
df["month_sin"] = np.sin(2 * np.pi * (df["month"] - 1) / 12)
df["month_cos"] = np.cos(2 * np.pi * (df["month"] - 1) / 12)

df[["hour", "hour_sin", "hour_cos", "month", "month_sin", "month_cos"]].head()


,hour,hour_sin,hour_cos,month,month_sin,month_cos
0,0,0.000000,1.000000,1,0.0,1.0
1,0,0.000000,1.000000,1,0.0,1.0
2,1,0.258819,0.965926,1,0.0,1.0
3,1,0.258819,0.965926,1,0.0,1.0
4,2,0.500000,0.866025,1,0.0,1.0


## 2. Feature Scaling (Normalization & Standardization)

Algorithms like **KNN**, **SVM**, and **Neural Networks** compute distances
or gradients directly on feature values. If one feature ranges 0-1 and
another ranges 0-10,000, the large-range feature will dominate the distance
calculation or the gradient updates — scaling puts every feature on a
comparable footing.

We'll scale the `reading` column (which contains outliers on purpose) with
all three scalers so you can compare their behavior side by side.

### 2-1. Standardization (Z-score Normalization)

Method used:

- **`StandardScaler`** — transforms each value to
  `z = (x - mean) / std`, giving the column mean 0 and standard deviation 1.
  Assumes/works best with roughly normally-distributed data; sensitive to
  outliers because both the mean and std are outlier-sensitive statistics.
  Key methods: `.fit()` learns mean/std from training data,
  `.transform()` applies them, `.fit_transform()` does both in one call.


In [5]:
standard_scaler = StandardScaler()
df["reading_standard_scaled"] = standard_scaler.fit_transform(df[["reading"]])

print("Mean after StandardScaler:", df["reading_standard_scaled"].mean().round(4))
print("Std  after StandardScaler:", df["reading_standard_scaled"].std().round(4))
df[["reading", "reading_standard_scaled"]].describe()


Mean after StandardScaler: 0.0
Std  after StandardScaler: 1.0025


,reading,reading_standard_scaled
count,200.000000,2.000000e+02
mean,50.846145,1.554312e-16
std,22.249198,1.002509e+00
min,-70.806429,-5.481449e+00
25%,46.421377,-1.993722e-01
50%,49.979041,-3.907020e-02
75%,52.768172,8.660310e-02
max,202.609708,6.838197e+00


### 2-2. Min-Max Scaling (Normalization)

Method used:

- **`MinMaxScaler`** — rescales each value to a fixed range (default
  `[0, 1]`) using `x_scaled = (x - min) / (max - min)`. Great when you need
  bounded output (e.g. for neural network inputs), but very sensitive to
  outliers: a single extreme value stretches the whole range and compresses
  the rest of the data toward one end.


In [6]:
minmax_scaler = MinMaxScaler()  # default feature_range=(0, 1)
df["reading_minmax_scaled"] = minmax_scaler.fit_transform(df[["reading"]])

print("Min after MinMaxScaler:", df["reading_minmax_scaled"].min())
print("Max after MinMaxScaler:", df["reading_minmax_scaled"].max())
df[["reading", "reading_minmax_scaled"]].describe()


Min after MinMaxScaler: 0.0
Max after MinMaxScaler: 0.9999999999999998


,reading,reading_minmax_scaled
count,200.000000,200.000000
mean,50.846145,0.444936
std,22.249198,0.081375
min,-70.806429,0.000000
25%,46.421377,0.428752
50%,49.979041,0.441764
75%,52.768172,0.451965
max,202.609708,1.000000


### 2-3. Robust Scaling

Method used:

- **`RobustScaler`** — centers using the **median** and scales using the
  **IQR** (interquartile range, i.e. the 75th percentile minus the 25th
  percentile) instead of mean/std. Because median and IQR are not pulled
  around by extreme values, this scaler is the right choice whenever the
  dataset has significant outliers — exactly the situation we engineered
  into the `reading` column.


In [7]:
robust_scaler = RobustScaler()
df["reading_robust_scaled"] = robust_scaler.fit_transform(df[["reading"]])

# Compare how each scaler treats the injected outliers
comparison = df.loc[outlier_idx, ["reading", "reading_standard_scaled",
                                   "reading_minmax_scaled", "reading_robust_scaled"]]
comparison.sort_values("reading")


,reading,reading_standard_scaled,reading_minmax_scaled,reading_robust_scaled
104,-70.806429,-5.481449,0.000000,-19.030940
17,-68.428763,-5.374316,0.008696,-18.656316
79,190.062155,6.272827,0.954108,22.071474
174,201.383454,6.782944,0.995515,23.855257
148,202.609708,6.838197,1.000000,24.048465


## 3. Integration into ML Pipelines

### 3-1. Encoding Categorical Features

The temporal/categorical components we extracted (like `device_type`, or a
derived `month` name) are categorical, not numeric — we must encode them.

Methods used:

- **`OneHotEncoder`** — creates one binary (0/1) column per category. Use
  this when categories have **no inherent order** (e.g. `device_type`).
  `handle_unknown="ignore"` prevents errors if a new category appears at
  prediction time.
- **`OrdinalEncoder`** — maps each category to a single integer
  (0, 1, 2, ...). Use this only when categories **do** have a natural order
  (e.g. `day_type`: Weekday < Weekend doesn't really have order, but
  something like Low < Medium < High would).


In [8]:
# Example categorical feature: whether the timestamp falls on a weekend
df["day_type"] = np.where(df["weekday"] >= 5, "Weekend", "Weekday")

# --- OneHotEncoder: no ordinal relationship between device types ---
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
device_encoded = ohe.fit_transform(df[["device_type"]])
device_encoded_df = pd.DataFrame(device_encoded, columns=ohe.get_feature_names_out(["device_type"]))
print("One-hot encoded device_type:")
display(device_encoded_df.head())

# --- OrdinalEncoder: forcing an explicit order Weekday < Weekend for illustration ---
ordinal = OrdinalEncoder(categories=[["Weekday", "Weekend"]])
df["day_type_encoded"] = ordinal.fit_transform(df[["day_type"]])
df[["day_type", "day_type_encoded"]].drop_duplicates()


One-hot encoded device_type:


,device_type_HumiditySensor,device_type_PressureSensor,device_type_TempSensor
0,0.0,1.0,0.0
1,0.0,0.0,1.0
2,0.0,0.0,1.0
3,0.0,0.0,1.0
4,0.0,1.0,0.0


,day_type,day_type_encoded
0,Weekday,0.0
195,Weekend,1.0


### 3-2. Scikit-Learn Pipelines

Doing scaling/encoding "by hand" as above is fine for exploration, but it
risks **data leakage**: if you fit a scaler on the *entire* dataset before
splitting into train/test, information from the test set leaks into
training statistics (mean, std, min, max, median...).

The fix is to wrap every step into a single **`Pipeline`**, fit it only on
the training split, and let it apply the *exact same* learned
transformations to the test split.

Methods used:

- **`ColumnTransformer`** — applies different preprocessing to different
  columns in one object: numeric columns get a scaler, categorical columns
  get an encoder, all executed in parallel and concatenated into one output
  matrix.
- **`Pipeline`** — chains sequential steps (here: preprocessing →
  `RandomForestRegressor`) into a single estimator with one `.fit()` /
  `.predict()` interface. This is what actually prevents leakage: calling
  `pipeline.fit(X_train, y_train)` fits the scaler/encoder using ONLY
  `X_train`, and `pipeline.predict(X_test)` reuses those fitted parameters.


In [9]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

# Feature set: engineered temporal features + categorical device type
feature_cols_numeric = ["hour_sin", "hour_cos", "month_sin", "month_cos", "day"]
feature_cols_categorical = ["device_type"]

X = df[feature_cols_numeric + feature_cols_categorical]
y = df["reading"]  # toy regression target: predict the sensor reading

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ColumnTransformer: RobustScaler for numeric (robust to the outliers we injected),
# OneHotEncoder for the categorical device_type column
preprocessor = ColumnTransformer(transformers=[
    ("num", RobustScaler(), feature_cols_numeric),
    ("cat", OneHotEncoder(handle_unknown="ignore"), feature_cols_categorical),
])

# Full pipeline: preprocessing -> model, fit/predict as a single unit
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(n_estimators=100, random_state=42)),
])

pipeline.fit(X_train, y_train)          # scaler/encoder learn ONLY from X_train
predictions = pipeline.predict(X_test)  # same learned params reused on X_test -> no leakage

mae = mean_absolute_error(y_test, predictions)
print(f"Pipeline fitted successfully. Test MAE: {mae:.3f}")
pipeline


Pipeline fitted successfully. Test MAE: 15.776


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

## Summary

| Step | Technique | Method(s) |
|---|---|---|
| 1-1 | Parse datetime strings | `pd.to_datetime`, `.dt` accessor |
| 1-2 | Manual extraction (teaching) | `str.split()` in a loop |
| 1-3 | Cyclical encoding | `np.sin`, `np.cos` |
| 2-1 | Standardization | `StandardScaler` |
| 2-2 | Min-Max scaling | `MinMaxScaler` |
| 2-3 | Robust scaling | `RobustScaler` |
| 3-1 | Categorical encoding | `OneHotEncoder`, `OrdinalEncoder` |
| 3-2 | Leak-proof orchestration | `ColumnTransformer`, `Pipeline` |
